In [ ]:
import os
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import config
import functions as func
import functions_data as dfunc
import mmpm as net   # 

config.config_gpu(0)

################################################################################
# DATASET
################################################################################
id_tau = [0,1,2,3,4,5,6,7,8,9,10,11]   # 
sigma = 'mix'
rescale_para = 400.0
signed = 0
type_task = 'T2mapping'

train_file = '/root/autodl-tmp//train/train.tfrecords'
val_file   = '/root/autodl-tmp//valid/valid.tfrecords'

dataset_train, dataset_valid, dataset_train_size, dataset_valid_size = \
    dfunc.load_data(
        train_file=train_file,
        val_file=val_file,
        id_tau=id_tau,
        rescale_para=rescale_para,
        type_task=type_task,
        signed=signed
    )



################################################################################
# MODEL
################################################################################
batch_size = 2
print('Create model ...')
Nc = len(id_tau)
Ns, Nk, Nt =4, 5, 1
ini_lam, ini_rho = 0.001, 0.1
Np, sep, f = 2, 0, 3
ini_lr     = 1e-4
epochs = 150
lr_decay = "cosine"   # 
# 
suffix = 'lam_{}_rho_{}_bs_{}_maemow_lr_{}_decay_{}'.format(
    ini_lam, ini_rho, batch_size, ini_lr, lr_decay
)

# 
model_dir = os.path.join(
    'model', type_task, '1.5Tmmpm_net',
    'mmpm_net_Ns_{}_Nk_{}_Nt_{}_Np_{}_signed_{}_sigma_{}_{}'.format(
        Ns, Nk, Nt, Np, signed, sigma, suffix
    )
)
os.makedirs(model_dir, exist_ok=True)

inpt_b  = tf.keras.layers.Input(shape=(None,None,Nc), name='images')
inpt_te = tf.keras.layers.Input(shape=(Nc,), name='tes')  # 

net = net.MMPM_Net(Ns=Ns, Nk=Nk, Nt=Nt, Np=Np, signed=signed,
                       f=f, ini_lam=ini_lam, ini_rho=ini_rho,
                       test_mode=False, name='MMPM_Net')

para_map = net([inpt_b, inpt_te])
model = tf.keras.Model(inputs=[inpt_b, inpt_te], outputs=para_map)

model.summary()
#######
# 
# ================== =================
print("\n" + "="*70)
print("MMPM网络内部层结构（用于微调）")
print("="*70)

# 
mmpm_net = model.get_layer('MMPM_Net')

# 
if hasattr(mmpm_net, 'layers'):
    print(f"\nMMPM_Net 包含 {len(mmpm_net.layers)} 个子层:\n")
    
    # 
    for i, layer in enumerate(mmpm_net.layers):
        layer_type = layer.__class__.__name__
        output_shape = str(layer.output_shape)
        
        # 
        extra_info = ""
        if hasattr(layer, 'trainable'):
            extra_info += f" | 可训练: {layer.trainable}"
        if hasattr(layer, 'kernel_size'):
            extra_info += f" | 核大小: {layer.kernel_size}"
        if hasattr(layer, 'filters'):
            extra_info += f" | 滤波器: {layer.filters}"
        
        print(f"[{i:3d}] {layer.name:40s}")
        print(f"     类型: {layer_type:20s} | 输出形状: {output_shape:30s}{extra_info}")
        
        # 
        if hasattr(layer, 'Ns'):
            print(f"     参数: Ns={layer.Ns}, Nk={layer.Nk}, Nt={layer.Nt}")
        
        #
        if hasattr(layer, 'weights') and layer.weights:
            for w in layer.weights:
                print(f"     权重: {w.name} -> {w.shape}")
        
        print()
else:
    print("MMPM_Net没有子层或layers属性不可访问")
    print("尝试使用 __dict__ 查看属性:")
    for key, value in mmpm_net.__dict__.items():
        if not key.startswith('_'):
            print(f"  {key}: {type(value).__name__}")
###########
# 
initial_epoch = func.findLastCheckpoint(save_dir=model_dir)
if initial_epoch > 0:
    print('Resuming by loading epoch %03d' % initial_epoch)
    model.load_weights(os.path.join(model_dir,'model_%03d.h5' % initial_epoch))

################################################################################
# LOSS & METRIC
################################################################################
opti    = tf.keras.optimizers.Adam(learning_rate=ini_lr)
loss    = func.MAE_mow()
metric  = [func.NRMSE_metric]
model.compile(optimizer=opti, loss=loss,metrics=metric)

################################################################################
# CALLBACKS
################################################################################
class EpochModelCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, filepath, save_freq=10, **kwargs):
        super().__init__()
        self.filepath = filepath
        self.save_freq = save_freq
        self.kwargs = kwargs

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.save_freq == 0:
            filepath = self.filepath.format(epoch=epoch + 1)
            self.model.save_weights(filepath, **self.kwargs)
            print(f"\nEpoch {epoch+1}: saving model to {filepath}")

checkpointer = EpochModelCheckpoint(
    os.path.join(model_dir, 'model_{epoch:03d}.h5'),
    save_freq=10,
    overwrite=True
)

csv_logger  = tf.keras.callbacks.CSVLogger(os.path.join(model_dir,'log.csv'), append=True, separator=',')
tensorboard = tf.keras.callbacks.TensorBoard(log_dir=model_dir, histogram_freq=1)

min_lr = 1e-6  # 可
total_epochs = epochs  # 
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(
    func.cosine_annealing(initial_lr=ini_lr, min_lr=min_lr, total_epochs=total_epochs)
)

################################################################################
# TRAIN
################################################################################
validation_batch_size = batch_size
dataset_train = dataset_train.batch(batch_size, drop_remainder=True).repeat()
dataset_valid = dataset_valid.batch(validation_batch_size, drop_remainder=False)

steps_per_epoch  = dataset_train_size // batch_size
validation_steps = dataset_valid_size // validation_batch_size
model.fit(
    x = dataset_train,
    epochs = epochs,
    steps_per_epoch = steps_per_epoch,
    validation_data = dataset_valid,
    validation_steps = validation_steps,
    initial_epoch = initial_epoch,
    callbacks = [checkpointer, tensorboard, lr_scheduler, csv_logger]
)

########################################################################
# PLOT
########################################################################
import matplotlib.pyplot as plt

log_file = os.path.join(model_dir, 'log.csv')
df = pd.read_csv(log_file)

print("日志列名:", df.columns.tolist())
epochs = df['epoch'] if 'epoch' in df.columns else df.index

plt.figure(figsize=(8,5))
plt.plot(epochs, df['loss'], label='Train Loss')
plt.plot(epochs, df['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
# plt.grid(True)
plt.savefig(os.path.join(model_dir, "loss_curve.png"), dpi=300)
plt.show()

# ================== =================
plt.figure(figsize=(8,5))
plt.plot(epochs, df['NRMSE_metric'], label='Train NRMSE')
plt.plot(epochs, df['val_NRMSE_metric'], label='Val NRMSE')
plt.xlabel('Epoch')
plt.ylabel('NRMSE')
plt.title('Training and Validation NRMSE')
plt.legend()
# plt.grid(True)
plt.savefig(os.path.join(model_dir, "nrmse_curve.png"), dpi=300)
plt.show()
